# More extensive EDA on dataset to see what needs claaning

In [0]:
df = spark.sql("SELECT * FROM marathos.bronze.raw_marathos")

df.display()

In [0]:
# Number of rows & columns

print(f"Number of rows: {df.count()}")
print(f"Number of columns: {len(df.columns)}")

In [0]:
print(df.columns)

In [0]:
df.printSchema()

In [0]:
display(df)

In [0]:
df.select("Event name").distinct().count()

In [0]:
df.select("Event name").distinct().display()
df.select("Year of event").distinct().display()
df.select("Athlete country").distinct().display()
df.select("Athlete gender").distinct().display()
df.select("Athlete club").distinct().display()
df.select("Event dates").distinct().display()

In [0]:
df.dtypes

In [0]:
df.select(
    [
        column
        for column, type_ in df.dtypes
        if type_ in ("int", "bigint", "double", "decimal")
    ]
).summary().display()

In [0]:
df.summary().display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
df.select("Athlete year of birth").distinct().display()
df.select("Athlete age category").distinct().display()

In [0]:
df.display()

In [0]:
df.groupBy("Athlete age category").count().orderBy("count", ascending=False).display()

In [0]:
%sql

SELECT `Athlete age category`, COUNT(*) as num_athletes
FROM marathos.bronze.raw_marathos
WHERE `Athlete age category` IS NOT NULL
GROUP BY `Athlete age category`
ORDER BY `Athlete age category` ASC

Databricks visualization. Run in Databricks to view.

In [0]:
country_counts = (
    df.groupBy("Athlete country")
    .count()
    .orderBy("count", ascending=False)
)

country_counts.limit(20).display()

Databricks visualization. Run in Databricks to view.

## Ingest new datasets
- Stockholm trail classic 2024
- country codes

Both are simulated by LLM (claude) and are based of Marathos dataset 

In [0]:
df_country = spark.sql("SELECT * FROM marathos.bronze.country_codes")

df_country.display()

In [0]:
df_stockholm = spark.sql("SELECT * FROM marathos.bronze.stockholm_trail_classic_2024")

df_stockholm.display()

# Cleaning Marathos dataset 

### Convert time of athlete performance to appropriate data type
I choose to save 'athlete performance' column as is for display but adding a column 'performance_seconds' for calculations 

In [0]:
from pyspark.sql.functions import col, regexp_extract, when

df = df.withColumn(
    "performance_seconds",
    when(
        col("Athlete performance").rlike(r"\d+:\d+:\d+"),
        regexp_extract(col("Athlete performance"), r"(\d+):(\d+):(\d+)", 1).cast("int") * 3600 +
        regexp_extract(col("Athlete performance"), r"(\d+):(\d+):(\d+)", 2).cast("int") * 60 +
        regexp_extract(col("Athlete performance"), r"(\d+):(\d+):(\d+)", 3).cast("int")
    ).otherwise(None)
)

df.display()

In [0]:
df.select("Event name", "Event distance/length", "Athlete performance").display()

In [0]:
df.select("Event distance/length").distinct().display()
df.select("Athlete performance").distinct().limit(100).display()

### Validate status of event and preformance
- if event has unit 'km' or 'mi¨' then preformance shold be in 'h'
- if event has unit h then preformance should be in 'km'
- remove all invalid rows (for simplicity you can consider d (days) as invalid and remove it)

In [0]:
from pyspark.sql.functions import col, when

df_validated = df.withColumn(
    "is_valid",
    when(
        # distance-event -> performace should end with 'h'
        col("Event distance/length").rlike("km|mi|mile"),
        col("Athlete performance").endswith("h")
    ).when(
        # time-event -> performace should end with 'km'
        col("Event distance/length").rlike("h"),
        col("Athlete performance").endswith("km")
    ).when(
        col("Event distance/length").endswith("d"),
        False
    ).otherwise(False)
)

df_validated.groupBy("is_valid").count().display()

df_validated.filter(col("is_valid") == False).select(
    "Event distance/length", "Athlete performance"
).limit(20).display()

### Create event_type column - distance/time

In [0]:
from pyspark.sql.functions import lit, when

df = df.withColumn(
    "event_type",
    when(col("Event distance/length").rlike("km|mi|mile"), lit("distance"))
    .when(col("Event distance/length").rlike("h"), lit("time"))
    .otherwise(None)
)

df.display()

### Rename columns to snake case 
Easier to work whit 

In [0]:
import re

def to_snake_case(name):
    # [\s]+ = en eller flera mellanrum byts ut mot ett understreck
    return re.sub(r"[\s]+", "_", name.strip().casefold())

# All columns
def rename_column_to_snake_case(df):
    new_column = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_column)

df_clean_column = rename_column_to_snake_case(df)
df_clean_column.display()

### Change data type on event dates -> to_date
I choose to add two new columns 'event_start_date' and 'event_end_date' to get the rows that contained multible dates e.g '03.-06.12.2018' and make them in to to_date datatype. I save the Event_date column for display but set all invalid dates to null - and then filter those out

In [0]:
df.select("Event dates").distinct().limit(10).display()

In [0]:
from pyspark.sql.functions import col, expr

df = df.withColumn(
    "event_start_date",
    expr("try_to_date(concat(regexp_extract(`Event dates`, '^(\\\\d{2})', 1), '.', regexp_extract(`Event dates`, '(\\\\d{2}\\\\.\\\\d{4})', 1)), 'dd.MM.yyyy')")
)

df = df.withColumn(
    "event_end_date",
    expr("try_to_date(regexp_extract(`Event dates`, '(\\\\d{2}\\\\.\\\\d{2}\\\\.\\\\d{4})$', 1), 'dd.MM.yyyy')")
)

# filter out all rows with invalid dates
df = df.filter(col("event_start_date").isNotNull())

df.select("Event dates", "event_start_date", "event_end_date").display()

### Change data type
- 'athlete_average_speed' -> double
- 'athlete_year_of_birth' -> integer 

In [0]:
df_clean_column = df_clean_column.withColumn("athlete_average_speed", col("athlete_average_speed").cast("double"))
df_clean_column = df_clean_column.withColumn("athlete_year_of_birth", col("athlete_year_of_birth").cast("integer"))

df_clean_column.printSchema()

In [0]:
# Number of rows
print("Number of rows:", df_clean_column.count())

# Number of unique athletes
print("Unique athletes:", df_clean_column.select("athlete_id").distinct().count())

# Avg result per athlete 
from pyspark.sql.functions import count

df_clean_column.groupBy("athlete_id").count().orderBy("count", ascending=False).limit(10).display()

### Create relevant id's
I use hashing (xxhash64) to generate id's - 'athlete_id' and 'result_id' 

In [0]:
from pyspark.sql.functions import col, xxhash64

# Event id based on event name
df_clean_column = df_clean_column.withColumn("event_id", xxhash64(col("event_name")))

# Restul id based on event name and athlete
df_clean_column = df_clean_column.withColumn("result_id", xxhash64(col("event_name"), col("athlete_id")))

df_clean_column.select("event_name", "event_id", "athlete_id", "result_id").display()

### Set values to 'unknown'
- athlete_club - an athlete doesn't have to belong to a club
- athlete_country - not neccesary

In [0]:
from pyspark.sql.functions import coalesce, lit

def fill_unknown(df, columns):
    def fill_unknown(df, columns):
        for c in columns:
            df = df.withColumn(c, coalesce(col(c).cast("string"), lit("Unknown")))
    return df

fill_columns = [
    "Athlete club",
    "Athlete country"
]

df = fill_unknown(df, fill_columns)

df.select("Athlete club", "Athlete country").display()

### Filter and set nulls to 0 for calculation
- athlete_average_speed - can be calculated from 'performance_seconds'
I filter out invalid values (fastest man in the world has average speed at 20.8km/h) and I don't think anyone walks slower than 2km/h at a marathon

In [0]:
from pyspark.sql.functions import coalesce, lit, expr

# try_cast handles trash values -> null insted of crash
df = df.withColumn(
    "Athlete average speed",
    expr("try_cast(`Athlete average speed` as double)")
)

# Filter invalid values
df = df.filter(
    col("Athlete average speed").isNull() |
    (
        (col("Athlete average speed") <= 20.8) &
        (col("Athlete average speed") >= 2.0)
    )
)

# Set 0 on nulls
df = df.withColumn(
    "Athlete average speed",
    coalesce(col("Athlete average speed"), lit(0.0))
)

df.select("Athlete average speed").display()

### Drop rows with null - rows is useless
- athlete_performance 
- event_start_date 
- athlete_id 
- athlete_gender 
- athlete_age_category 
- event_distance/length

In [0]:
from pyspark.sql.functions import col

def drop_null_rows(df, columns):
    for c in columns:
        df = df.filter(col(c).isNotNull())
    return df

# Columns with null = drop row
drop_columns = [
    "Athlete performance",
    "event_start_date",
    "Athlete ID",
    "Athlete gender",
    "Athlete age category",
    "Event distance/length",
]

df = drop_null_rows(df, drop_columns)

print(f"Number of rows: {df.count()}")


In [0]:
df.select("Year of event").distinct().orderBy("Year of event", ascending=False).display()

In [0]:
df.display()

### Filter out invalid age
I choose to set age limit at 18 and 100. 18 is the most common age limit worldwide and I don't think anyone above 100 years shold be running a marathon

In [0]:
from pyspark.sql.functions import col, year

df = df.filter(
    (year(col("event_start_date")) - col("Athlete year of birth") >= 18) &
    (year(col("event_start_date")) - col("Athlete year of birth") <= 100)
)
df.select("event_start_date", "Athlete year of birth").display()

In [0]:
df.select("Athlete gender").distinct().display()

df.select("Athlete age category").distinct().display()


### Change 'F' in 'Athlete age category' to 'W' (Woman)
I choose to change 'F' to 'W' to get the column uniformly (e.g WU23 - woman junior category)

In [0]:
from pyspark.sql.functions import col, regexp_replace

df = df.withColumn(
    "Athlete age category",
    regexp_replace(col("Athlete age category"), "^F", "W")  # F → W
)
df.select("Athlete age category").display()

### Clean Athlete club column
I notis in the gold layer that the club column was very dirty - I haven't prioritized it. 
I choose to only remove '*' for now and not spend more time on it now since I don't really think it's relevant fot the analyses. 

In [0]:
df = df.withColumn(
    "Athlete club",
    regexp_replace(col("Athlete club"), r"^\*", "")
)

df.select("Athlete club").display()

### Convert Athlete performance where it's 'km'

In [0]:
df.select("Athlete performance").display()

In [0]:
df = df.withColumn(
    "performance_km",
    when(
        col("Athlete performance").rlike(r"\d+\.?\d*\s*km"),
        regexp_extract(col("Athlete performance"), r"(\d+\.?\d*)", 1).cast("double"),
    ).otherwise(None),
)

df.select("performance_km").display()